# Exporting Modelica Models to FMUs

## Overview

This notebook demonstrates how to export Modelica models to FMUs using the `OMPython` library. The process involves identifying the Modelica models, compiling them into FMUs, and organizing the output files.

In [1]:
import sys
from typing import Literal
from pathlib import Path
from shutil import move

from OMPython import ModelicaSystem

from pathlib import Path

def find_repo_root(start: Path, marker: str = ".git") -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Could not find repo root containing '{marker}' from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
src_dir = REPO_ROOT / "demos" / "ControlledPendulum" / "src" / "modelica" / "ControlledPendulum"

PLATFORM = sys.platform
if PLATFORM.startswith("linux"):
    FMU_EXPORTER_PATH = REPO_ROOT / "demos/ControlledPendulum/artifacts/fmus/linux/"
elif PLATFORM.startswith("win"):
    FMU_EXPORTER_PATH = REPO_ROOT / f"demos/ControlledPendulum/artifacts/fmus/win32/"
elif PLATFORM.startswith("darwin"):
    FMU_EXPORTER_PATH = REPO_ROOT / "demos/ControlledPendulum/artifacts/fmus/macos/"
else:
    raise RuntimeError(f"Unsupported platform: {PLATFORM}")
FMU_EXPORTER_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
src_dir = REPO_ROOT / "demos/ControlledPendulum/src/modelica/ControlledPendulum/"
main_pkg_name = src_dir.name
main_pkg_path = src_dir /'package.mo'

In [3]:
sub_pkgs = {}
for item in src_dir.iterdir():
    if item.is_dir():
        sub_pkgs[item.name] = {}
        for sub_sub_pkg in item.iterdir():
            if sub_sub_pkg.is_dir():
                sub_pkgs[item.name][sub_sub_pkg.name] = sub_sub_pkg
        if not sub_pkgs[item.name]:
            sub_pkgs[item.name] = item

sub_pkgs

{'Actuators': WindowsPath('C:/Users/flori/source/repos/FlorianFrech/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Actuators'),
 'Controllers': WindowsPath('C:/Users/flori/source/repos/FlorianFrech/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Controllers'),
 'Examples': {'Contact': WindowsPath('C:/Users/flori/source/repos/FlorianFrech/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Examples/Contact'),
  'NoContact': WindowsPath('C:/Users/flori/source/repos/FlorianFrech/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Examples/NoContact')},
 'Plants': WindowsPath('C:/Users/flori/source/repos/FlorianFrech/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Plants'),
 'Sensors': WindowsPath('C:/Users/flori/source/repos/FlorianFrech/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Sensors'),
 'Trajectories': WindowsPath('C:/Users/flori/so

In [4]:
model_names = {}
for pkg_name, pkg in sub_pkgs.items():
    if isinstance(pkg, dict):
        model_names[pkg_name] = {}
        for sub_name, sub_pkg in pkg.items():
            model_names[pkg_name][sub_name] = {}
            for item in sub_pkg.iterdir():
                if item.is_file() and item.suffix == ".mo" and item.stem != "package":
                    model_name = f"{main_pkg_name}.{pkg_name}.{sub_name}.{item.stem}"
                    model_names[pkg_name][sub_name][item.stem] = model_name
    else:
        model_names[pkg_name] = {}
        for item in pkg.iterdir():
            if item.is_file() and item.suffix == ".mo" and item.stem != "package":
                model_name = f"{main_pkg_name}.{pkg_name}.{item.stem}"
                model_names[pkg_name][item.stem] = model_name

In [5]:
model_names

{'Actuators': {'DriveAdvanced': 'ControlledPendulum.Actuators.DriveAdvanced',
  'DriveDynamic': 'ControlledPendulum.Actuators.DriveDynamic',
  'DriveSimple': 'ControlledPendulum.Actuators.DriveSimple'},
 'Controllers': {'PIDController': 'ControlledPendulum.Controllers.PIDController',
  'PIDControllerReset': 'ControlledPendulum.Controllers.PIDControllerReset'},
 'Examples': {'Contact': {'CompliantContact': 'ControlledPendulum.Examples.Contact.CompliantContact',
   'RigidContact': 'ControlledPendulum.Examples.Contact.RigidContact'},
  'NoContact': {'AlgebraicLoop': 'ControlledPendulum.Examples.NoContact.AlgebraicLoop',
   'Baseline': 'ControlledPendulum.Examples.NoContact.Baseline',
   'Quantization': 'ControlledPendulum.Examples.NoContact.Quantization'}},
 'Plants': {'ImpactWallCompliant': 'ControlledPendulum.Plants.ImpactWallCompliant',
  'ImpactWallDiscrete': 'ControlledPendulum.Plants.ImpactWallDiscrete',
  'Pendulum': 'ControlledPendulum.Plants.Pendulum',
  'PendulumAnimationNoWall'

In [6]:
SOLVER = Literal["cvode", "euler"]
def create_fmu(package_file_path: Path,
               composed_model_name: str,
               solver: SOLVER,
               export_path: Path):
    """Create a ModelicaSystem instance for a given package and model.
    
    Args:
        package_file_path (Path): Path to the Modelica main package file (package.mo).
        composed_model_name (str): Name of the model to be instantiated (e.g., "MainPackageName.SubPackageName.ModelName").
        solver (LiteralString): The solver to be used for simulation (e.g., "cvode", "euler").
    Returns:
        ModelicaSystem: An instance of ModelicaSystem for the specified model.
    """
    try:
        modelica_system = ModelicaSystem(
            fileName=str(package_file_path),
            modelName=composed_model_name,
            commandLineOptions=[f"--fmiFlags=s:{solver}"]
        )

        modelica_system.buildModel()

        fmu_path = modelica_system.convertMo2Fmu(version="2.0", fmuType="cs")
    
        move(fmu_path, export_path)
        print(f"FMU created at: {export_path}")
    except Exception as e:
        print(f"Error moving FMU: {e}")

In [7]:
# Define Models that shall also be compiled with Euler solver
euler_models = ["PIDControllerReset", "Pendulum", "AngleSensor", "AngleDecoder"]

In [9]:
for pkg_name, dir in model_names.items():
    export_dir = FMU_EXPORTER_PATH / pkg_name
    export_dir.mkdir(parents=True, exist_ok=True)

    for sub_pkg_name, sub_item in dir.items():
        if isinstance(sub_item, dict):
            for model_name, composed_model_name in sub_item.items():
                print(100 * '=')
                print(f"Creating FMU for model: {composed_model_name}")
                if model_name in euler_models:
                    create_fmu(main_pkg_path, composed_model_name, "euler", export_dir / f"{model_name}_euler.fmu")
                    create_fmu(main_pkg_path, composed_model_name, "cvode", export_dir / f"{model_name}_cvode.fmu")
                else:
                    solver = "cvode"
                    create_fmu(main_pkg_path, composed_model_name, solver, export_dir / f"{model_name}.fmu")
        else:
            model_name = sub_pkg_name
            composed_model_name = sub_item
            print(100 * '=')
            print(f"Creating FMU for model: {composed_model_name}")
            if model_name in euler_models:
                create_fmu(main_pkg_path, composed_model_name, "euler", export_dir / f"{model_name}_euler.fmu")
                create_fmu(main_pkg_path, composed_model_name, "cvode", export_dir / f"{model_name}_cvode.fmu")
            else:
                solver = "cvode"
                create_fmu(main_pkg_path, composed_model_name, solver, export_dir / f"{model_name}.fmu")

Creating FMU for model: ControlledPendulum.Actuators.DriveAdvanced
Error moving FMU: 'NoneType' object is not subscriptable
Creating FMU for model: ControlledPendulum.Actuators.DriveDynamic
Error moving FMU: 'NoneType' object is not subscriptable
Creating FMU for model: ControlledPendulum.Actuators.DriveSimple
Error moving FMU: 'NoneType' object is not subscriptable
Creating FMU for model: ControlledPendulum.Controllers.PIDController
Error moving FMU: 'NoneType' object is not subscriptable
Creating FMU for model: ControlledPendulum.Controllers.PIDControllerReset
Error moving FMU: 'NoneType' object is not subscriptable
Error moving FMU: 'NoneType' object is not subscriptable
Creating FMU for model: ControlledPendulum.Examples.Contact.CompliantContact
Error moving FMU: 'NoneType' object is not subscriptable
Creating FMU for model: ControlledPendulum.Examples.Contact.RigidContact
Error moving FMU: 'NoneType' object is not subscriptable
Creating FMU for model: ControlledPendulum.Examples.N

In [8]:
export_dir = FMU_EXPORTER_PATH / 'Sensors'
export_dir.mkdir(parents=True, exist_ok=True)
model_name = model_names['Sensors']['AngleSensor']
create_fmu(main_pkg_path, model_name, "cvode", export_dir / f"AngleSensor.fmu")
model_name = model_names['Sensors']['AngleDecoder']
create_fmu(main_pkg_path, model_name, "cvode", export_dir / f"AngleDecoder.fmu")

FMU created at: C:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\artifacts\fmus\win32\Sensors\AngleSensor.fmu
FMU created at: C:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\artifacts\fmus\win32\Sensors\AngleDecoder.fmu


In [19]:
export_dir = FMU_EXPORTER_PATH / 'Controllers'
export_dir.mkdir(parents=True, exist_ok=True)
model_name = model_names['Controllers']['PIDControllerReset']
create_fmu(main_pkg_path, model_name, "euler", export_dir / f"PIDControllerReset_euler.fmu")
create_fmu(main_pkg_path, model_name, "cvode", export_dir / f"PIDControllerReset_cvode.fmu")

FMU created at: C:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\artifacts\fmus\windows\Controllers\PIDControllerReset_euler.fmu
FMU created at: C:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\artifacts\fmus\windows\Controllers\PIDControllerReset_cvode.fmu
